In [6]:
import numpy as np
import tensorflow as tf
import os
import zipfile
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

TensorFlow version: 2.19.0
NumPy version: 2.0.2


In [7]:
# Download dataset SelfBack
url = "https://archive.ics.uci.edu/static/public/521/selfback.zip"
zip_path = "selfback.zip"
extract_dir = "."
dataset_path = "selfBACK"

if not os.path.exists(dataset_path):
    print("Downloading SelfBack dataset (~94 MB)...")
    urllib.request.urlretrieve(url, zip_path)
    print("Download completato. Estrazione...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    os.remove(zip_path)
    print("Estrazione completata!")
else:
    print("Dataset gia' presente.")


# filtraggio di eventuali file non necessari come file macOS
# inoltre conta quanti file sono presenti
wrist_path = os.path.join(dataset_path, "w")
print(f"\nAttività disponibili nella cartella polso (w/):")
activities_raw = sorted([
    entry.name
    for entry in os.scandir(wrist_path)
    if entry.is_dir() and not entry.name.startswith('.')
])
for act in activities_raw:
    files = [
        f for f in os.listdir(os.path.join(wrist_path, act))
        if f.endswith('.csv') and not f.startswith('.')
    ]
    print(f"  {act}: {len(files)} partecipanti")

Dataset gia' presente.

Attività disponibili nella cartella polso (w/):
  downstairs: 33 partecipanti
  jogging: 33 partecipanti
  lying: 33 partecipanti
  sitting: 33 partecipanti
  standing: 33 partecipanti
  upstairs: 33 partecipanti
  walk_fast: 33 partecipanti
  walk_mod: 33 partecipanti
  walk_slow: 33 partecipanti


In [8]:
SAMPLE_RATE_HZ  = 100    # frequenza dati in dataset SelfBack
WINDOW_SIZE     = 200    # campioni per finestra, cioè 2.0 secondi a 100Hz
STEP_SIZE       = 100    # passo tra finestre di 100 per ottenere overlap 50%
N_FEATURES      = 3      # tre assi di misurazione



# accorpamento delle 9 classi del dataset in 4 classi, serve per caricare i dati
ACTIVITY_MAP = {
    'upstairs':   'WALKING',
    'downstairs': 'WALKING',
    'walk_slow':  'WALKING',
    'walk_mod':   'WALKING',
    'walk_fast':  'WALKING',
    'jogging':    'JOGGING',
    'standing':   'STANDING',
    'sitting':    'SEDENTARY',
    'lying':      'SEDENTARY',
}

# creiamo una lista con le solo 4 attività da riconoscere, messe in ordine a partire da ACTIVITY_MAP e
# rimuovendo i duplicati, ci servirà per capire l'indice restituito a che attivtà corrisponde
ACTIVITIES = list(dict.fromkeys(ACTIVITY_MAP.values()))
N_CLASSES = len(ACTIVITIES)

In [18]:
# carica un sinoglo csv per ogni partecipante e ne estrae i segnali
def load_participant_signal(filepath):
    try:
        df = pd.read_csv(filepath, header=0)
        # selezioniamo solo le colonne con i dati dell'accelermetro
        signal = df.iloc[:, 1:4].values.astype(np.float32)
        return signal
    except Exception as e:
        print(f"  Errore caricamento {filepath}: {e}")
        return None


# creaiamo i sample effettivi che daremo in pasto al modello
# partiamo da 0, avanziamo di 100 alla volta e per ogni
# passo estraiamo 200 segnali, con un overlap del 50%
# se segnale troppo corto, ritorna array vuoto
def apply_sliding_window(signal, window_size, step_size):
    windows = []
    n_samples = len(signal)
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        windows.append(signal[start:end])
    return np.array(windows) if windows else np.empty((0, window_size, signal.shape[1]))



def load_all_data(wrist_path, activity_map, window_size, step_size, test_participant_ids):
    X_train, y_train = [], []
    X_test,  y_test  = [], []

    # estraggo tutti in una lista tutti i valori del dizionario, rimuovendo i duplicati
    activities_list = list(dict.fromkeys(activity_map.values()))
    # creo un dizionario nome_cartella : indice dell'attività corrispondente
    folder_to_idx = {}
    for folder, class_name in activity_map.items():
        folder_to_idx[folder] = activities_list.index(class_name)

    # apro ogni cartella
    for folder_name in activity_map.keys():
        class_idx = folder_to_idx[folder_name]
        class_name = activity_map[folder_name]
        folder_path = os.path.join(wrist_path, folder_name)
        if not os.path.isdir(folder_path):
            print(f"  ATTENZIONE: cartella non trovata: {folder_path}")
            continue

        # tolgo tutti i file che non ci servono e tengo solo csv
        files = sorted([
            f for f in os.listdir(folder_path)
            if f.endswith('.csv') and not f.startswith('.')
        ])

        n_windows_class = 0

        for fname in files:
            participant_id = int(os.path.splitext(fname)[0])

            fpath = os.path.join(folder_path, fname)
            # estraggo i segnali
            signal = load_participant_signal(fpath)
            if signal is None or len(signal) < window_size:
                continue

            # faccio sliding windows
            windows = apply_sliding_window(signal, window_size, step_size)
            if len(windows) == 0:
                continue

            # creo array di etichette
            labels = np.full(len(windows), class_idx, dtype=np.int32)
            n_windows_class += len(windows)

            # divido in training e test set
            if participant_id in test_participant_ids:
                X_test.append(windows)
                y_test.append(labels)
            else:
                X_train.append(windows)
                y_train.append(labels)

        print(f"  {folder_name} -> {class_name} (idx={class_idx}): {n_windows_class} finestre")

    X_train = np.concatenate(X_train, axis=0)
    y_train = np.concatenate(y_train, axis=0)
    X_test  = np.concatenate(X_test,  axis=0)
    y_test  = np.concatenate(y_test,  axis=0)

    return X_train, y_train, X_test, y_test



# leggo id dei partecipanti e li divido in training e test set
# primi 80% in training set, restante 20% in test set
first_activity = list(ACTIVITY_MAP.keys())[0]
first_folder = os.path.join(wrist_path, first_activity)
all_ids = sorted([
    int(os.path.splitext(f)[0])
    for f in os.listdir(first_folder)
    if f.endswith('.csv') and not f.startswith('.')
])
TEST_PARTICIPANTS = set(all_ids[-7:])
TRAIN_PARTICIPANTS = set(all_ids[:-7])


X_train, y_train, X_test, y_test = load_all_data(
    wrist_path=wrist_path,
    activity_map=ACTIVITY_MAP,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    test_participant_ids=TEST_PARTICIPANTS
)


  upstairs -> WALKING (idx=0): 2568 finestre
  downstairs -> WALKING (idx=0): 2352 finestre
  walk_slow -> WALKING (idx=0): 5896 finestre
  walk_mod -> WALKING (idx=0): 5769 finestre
  walk_fast -> WALKING (idx=0): 5863 finestre
  jogging -> JOGGING (idx=1): 7799 finestre
  standing -> STANDING (idx=2): 5940 finestre
  sitting -> SEDENTARY (idx=3): 5766 finestre
  lying -> SEDENTARY (idx=3): 5869 finestre


In [10]:
# mescolo training set mantenendo la corrispondenza x-y
np.random.seed(42)
indices = np.random.permutation(len(X_train))
X_train = X_train[indices]
y_train = y_train[indices]

In [19]:
n_timesteps = X_train.shape[1]  # 200
n_features  = X_train.shape[2]  # 3
n_classes   = N_CLASSES          # 9


# creazione modello
model = tf.keras.Sequential([
    # layer1
    tf.keras.layers.Conv1D(filters=64, kernel_size=5, activation='relu',
                           input_shape=(n_timesteps, n_features)),
    tf.keras.layers.MaxPool1D(pool_size=2),

    # layer2
    tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation='relu'),
    tf.keras.layers.MaxPool1D(pool_size=2),

    # layer3
    tf.keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu'),
    tf.keras.layers.MaxPool1D(pool_size=2),

    # layer 4
    tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu'),

    # layer 5
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(n_classes, activation='softmax')
])


# configuro ottimizzatore
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 196, 64)        │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 98, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 94, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 45, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 22, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 20, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 280,516 (1.07 MB)

 Trainable params: 280,516 (1.07 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# meccanismo che interrompe il training quando il modello smette di migliorare
# ripristina i pesi a quelli della versione migliore
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)


print("Inizio training...\n")

history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining completato!")

Inizio training...

Epoch 1/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9353 - loss: 0.1981 - val_accuracy: 0.9662 - val_loss: 0.1173
Epoch 2/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9763 - loss: 0.0897 - val_accuracy: 0.9600 - val_loss: 0.1279
Epoch 3/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9825 - loss: 0.0713 - val_accuracy: 0.9637 - val_loss: 0.1129
Epoch 4/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9834 - loss: 0.0651 - val_accuracy: 0.9603 - val_loss: 0.1477
Epoch 5/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9846 - loss: 0.0603 - val_accuracy: 0.9775 - val_loss: 0.0843
Epoch 6/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9862 - loss: 0.0520 - val_accuracy: 0.9678 - val_loss: 0.1207
Epoch 7/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9857 - loss: 0.0530 - val_accuracy: 0.9674 - val_loss: 0.1354
Epoch 8/40
592/592 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9866 - loss: 0.0

In [13]:
# accuratezza modello
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)

print(f"\n--- Confusion Matrix ---")
short_names = [a[:8] for a in ACTIVITIES]
print(f"{'':>22}", end='')
for name in short_names:
    print(f"{name:>10}", end='')
print()
for i, act in enumerate(ACTIVITIES):
    print(f"{act:>22}", end='')
    for j in range(n_classes):
        count = np.sum((y_test == i) & (y_pred == j))
        print(f"{count:>10}", end='')
    print()

print(f"\n--- Accuracy per Attività ---")
for i, act in enumerate(ACTIVITIES):
    mask = y_test == i
    if np.sum(mask) > 0:
        acc = np.sum(y_pred[mask] == i) / np.sum(mask) * 100
        print(f"  {act}: {acc:.1f}% ({np.sum(mask)} campioni)")

Test Accuracy: 97.75%
Test Loss: 0.0843
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

--- Confusion Matrix ---
                         WALKING   JOGGING  STANDING  SEDENTAR
               WALKING      4782         7        10         0
               JOGGING        36      1382         1         4
              STANDING        70         0      1189        37
             SEDENTARY        21         1        37      2374

--- Accuracy per Attività ---
  WALKING: 99.6% (4799 campioni)
  JOGGING: 97.1% (1423 campioni)
  STANDING: 91.7% (1296 campioni)
  SEDENTARY: 97.6% (2433 campioni)


In [20]:
# salvataggio modello formato keras
model.save('har_model.h5')
print("Modello Keras salvato: har_model.h5")


# conversione modello normale float32
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = 'har_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"\nModello TFLite (float32): {tflite_path}")
print(f"Dimensione: {os.path.getsize(tflite_path) / 1024:.1f} KB")


# conversione modello quantizzato
converter_q = tf.lite.TFLiteConverter.from_keras_model(model)
# quantizzazione post training
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_q = converter_q.convert()

tflite_q_path = 'har_model_quantized.tflite'
with open(tflite_q_path, 'wb') as f:
    f.write(tflite_model_q)
print(f"\nModello TFLite (quantizzato): {tflite_q_path}")
print(f"Dimensione: {os.path.getsize(tflite_q_path) / 1024:.1f} KB")

Modello Keras salvato: har_model.h5
Saved artifact at '/tmp/tmpfzrzb5jj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 3), dtype=tf.float32, name='keras_tensor_12')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  132282356368208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356369168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356370128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356369360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356370512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356370320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356370896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356370704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356371280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132282356371088: TensorSpec(shape=(), dtype=tf.res

In [17]:
# salvataggio file label
with open('har_labels.txt', 'w') as f:
    for act in ACTIVITIES:
        f.write(act + '\n')
print("File har_labels.txt creato:")
for i, act in enumerate(ACTIVITIES):
    print(f"  {i}: {act}")

# download
try:
    from google.colab import files
    print("\nDownload dei file...")
    files.download('har_model.tflite')
    files.download('har_model_quantized.tflite')
    files.download('har_labels.txt')
    print("Download completato!")
except ImportError:
    print("\nNon sei su Google Colab.")
    print("I file sono disponibili nella directory corrente:")
    print(f"  - har_model.tflite ({os.path.getsize('har_model.tflite') / 1024:.1f} KB)")
    print(f"  - har_model_quantized.tflite ({os.path.getsize('har_model_quantized.tflite') / 1024:.1f} KB)")
    print(f"  - har_labels.txt")

File har_labels.txt creato:
  0: WALKING
  1: JOGGING
  2: STANDING
  3: SEDENTARY

Download dei file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download completato!
